In [1]:
import pandas as pd
import numpy as np
import time
import math
import os.path

from tqdm import tqdm
from datetime import timedelta, datetime
from dateutil import parser
import json
from binance.client import Client
binsizes = {"1m":1,"5m":5,"15m":15,"1h":60,"1d":1440, "funding":480}
binance_client = Client()
data_folder = "/Users/chinjieheng/Documents/data/binance_fundingrate_data/"
os.makedirs(data_folder, exist_ok=True)

def minutes_of_new_data(symbol, kline_size, data, source):
    """Fixed logic errors in timestamp handling"""
    
    if len(data) > 0:  
        # ERROR 1: Need to handle indexed DataFrame properly
        if isinstance(data.index, pd.DatetimeIndex):
            # Data has timestamp as index (after set_index)
            old = data.index[-1]
        elif 'fundingTime' in data.columns:
            # Data has timestamp as column
            old = parser.parse(str(data["fundingTime"].iloc[-1]))
        else:
            # Fallback if no timestamp found
            old = datetime.strptime('1 Jan 2022', '%d %b %Y')
    elif source == "binance": 
        old = datetime.strptime('1 Jan 2022', '%d %b %Y')
    else:
        # ERROR 2: No fallback for non-binance sources
        old = datetime.now()

    if source == "binance": 
        # ERROR 3: Should handle API errors
        try:
            new = pd.to_datetime(binance_client.futures_funding_rate(symbol=symbol, limit=1)[0]['fundingTime'], unit='ms')
        except Exception as e:
            print(f"Error getting latest timestamp: {e}")
            new = datetime.now()
    else:
        # ERROR 4: No handling for non-binance sources
        new = datetime.now()

    return old, new

def add_interval_hours(df):
    """Infer funding interval (hours) from consecutive fundingTime index values."""
    if len(df) == 0:
        return df
    df = df.sort_index()
    if len(df) == 1:
        df['fundingIntervalHours'] = pd.Series([pd.NA], index=df.index, dtype='Int64')
        return df
    
    # Calculate raw diffs in hours
    hours = df.index.to_series().diff().dt.total_seconds() / 3600.0
    
    # Use forward fill as requested
    hours = hours.ffill()
    
    # Backfill the first row with the second row
    if len(hours) > 1:
        hours.iloc[0] = hours.iloc[1]
        
    hours = hours.round()
    df['fundingIntervalHours'] = hours.astype('Int64')
    return df

def get_all_binance(symbol, kline_size, save = False):
    try:
            
        filename = data_folder + '%s-%s-data.parquet' % (symbol, kline_size)
        
        if os.path.isfile(filename): 
            data_df = pd.read_parquet(filename)
            # FIX: Remove 'ignore' column from existing data if it exists
            if 'ignore' in data_df.columns:
                data_df = data_df.drop('ignore', axis=1)
            # FIX: Convert existing data columns to numeric (they might be strings)
            numeric_columns = ['fundingRate']
            for col in numeric_columns:
                if col in data_df.columns:
                    data_df[col] = pd.to_numeric(data_df[col], errors='coerce')
        else: 
            data_df = pd.DataFrame()
            
        oldest_point, newest_point = minutes_of_new_data(symbol, kline_size, data_df, source = "binance")
        delta_min = (newest_point - oldest_point).total_seconds()/60
        available_data = math.ceil(delta_min/binsizes[kline_size])
        
        if oldest_point == datetime.strptime('1 Jan 2022', '%d %b %Y'): 
            print('Downloading all available %s data for %s. Be patient..!' % 
                (kline_size, symbol))
        else: 
            print('Downloading %d minutes of new data available for %s, i.e. %d instances of %s data.' % 
                (delta_min, symbol, available_data, kline_size))
            
        # Fetch funding rates in a loop to handle pagination
        funding_rates = []
        start_ts = int(oldest_point.timestamp() * 1000)
        while True:
            try:
                new_rates = binance_client.futures_funding_rate(symbol=symbol, startTime=start_ts, limit=1000)
                if not new_rates:
                    break
                funding_rates.extend(new_rates)
                if len(new_rates) < 1000:
                    break
                start_ts = new_rates[-1]['fundingTime'] + 1
            except Exception as e:
                print(f"Error fetching funding rates: {e}")
                break
        
        data = pd.DataFrame(funding_rates)
        if 'fundingTime' in data.columns:
            data['fundingTime'] = pd.to_datetime(data['fundingTime'], unit='ms')
            data.set_index('fundingTime', inplace=True)
        # Convert numeric columns for new data
        numeric_columns = ['fundingRate']
        for col in numeric_columns:
            if col in data.columns:
                data[col] = pd.to_numeric(data[col], errors='coerce')
        # Drop unnecessary columns if present
        if 'symbol' in data.columns:
            data = data.drop('symbol', axis=1)
        
        if len(data_df) > 0:
            combined_data = pd.concat([data_df, data])
            combined_data = combined_data[~combined_data.index.duplicated(keep='last')]
            combined_data = combined_data.sort_index()
            data_df = combined_data
        else: 
            data_df = data
            
        # Infer interval hours from consecutive timestamps only
        data_df = add_interval_hours(data_df)
        
        if save: 
            data_df.to_parquet(filename)
            
        print('All caught up..!')
        return data_df
    
    except Exception as e:
        print(f'ERROR in {symbol}: {str(e)}')
        import traceback
        traceback.print_exc()  # This will show the full error
        raise  # Re-raise to see the actual error

info = binance_client.futures_exchange_info()

symbols = info['symbols']
coins = []
others = []

for i, symbol in enumerate(symbols):
    # Filter for perpetual contracts only
    if symbol['contractType'] != 'PERPETUAL':
        continue
    s = symbol['symbol']
    if ('USDT' in s):
#         print('{} - {}'.format(i, s))
        coins.append(s)


for symbol in tqdm(coins):
    try:
        get_all_binance(symbol, 'funding', save=True)
    except Exception as e:
        print(f'Skipping {symbol}: {str(e)}')  # Show the actual error message
        pass

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 1/625 [00:00<06:54,  1.50it/s]

All caught up..!


  0%|          | 2/625 [00:01<10:47,  1.04s/it]

All caught up..!


...

All caught up..!


100%|█████████▉| 624/625 [03:48<00:00,  3.08it/s]

All caught up..!


100%|██████████| 625/625 [03:48<00:00,  2.73it/s]

All caught up..!
